In [3]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, Subset, ConcatDataset
from torchvision import datasets, transforms, models
import numpy as np

In [4]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
BATCH_SIZE = 64
CONFIDENCE_THRESHOLD = 0.95  # Impact factor 1: Quality vs. Quantity
LABEL_RATIO = 0.1

In [5]:
transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])
full_train_set = datasets.CIFAR10(root='./data', train=True, download=True, transform=transform)
test_set = datasets.CIFAR10(root='./data', train=False, download=True, transform=transform)


100%|██████████| 170M/170M [00:18<00:00, 8.98MB/s]


In [6]:
num_train = len(full_train_set)
indices = list(range(num_train))
np.random.shuffle(indices)
split = int(np.floor(LABEL_RATIO * num_train))

labeled_idx, unlabeled_idx = indices[:split], indices[split:]
labeled_set = Subset(full_train_set, labeled_idx)
unlabeled_set = Subset(full_train_set, unlabeled_idx)

In [7]:
def get_model():
    model = models.resnet18(weights='DEFAULT')
    model.fc = nn.Linear(model.fc.in_features, 10)  # Adjust for 10 classes
    return model.to(device)

In [8]:
def get_pseudo_labels(model, unlabeled_loader, threshold):
    model.eval()
    pseudo_data = []
    with torch.no_grad():
        for inputs, _ in unlabeled_loader:
            inputs = inputs.to(device)
            outputs = model(inputs)
            probs = torch.softmax(outputs, dim=1)
            max_probs, targets = torch.max(probs, dim=1)

            # Filter by confidence threshold
            mask = max_probs > threshold
            if mask.any():
                # In a real scenario, you'd store these as a new Dataset
                # This is a conceptual simplification
                pass
    return pseudo_data

In [9]:
model = get_model()
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=1e-4)

Downloading: "https://download.pytorch.org/models/resnet18-f37072fd.pth" to /root/.cache/torch/hub/checkpoints/resnet18-f37072fd.pth


100%|██████████| 44.7M/44.7M [00:00<00:00, 180MB/s]
